# AtlasRank M5 L4 training pilot — preparation only

All cells are unexecuted. This notebook refuses to train unless a separate reviewed manifest has execution_approved=true, pins this exact training export and source snapshot, names a private Drive destination, and carries a unique owner token. The first run is one deterministic pass over 9,180 train-only groups; no development labels enter training.

Before running: review docs/reviews/M5-preparation.md and configs/m5-colab-pilot.proposed.json; select Colab's L4; note its UTC allocation time before uploads/setup; confirm the private Drive account and empty M5 directory; upload the approved manifest and exact pinned ZIP.



In [ ]:
from pathlib import Path

ALLOCATION_STARTED_UTC = None  # Actual UTC epoch from L4 allocation, before upload/setup.
ARCHIVE_UPLOAD = Path("/content/atlasrank-m5-training-input-v3.zip")
APPROVED_MANIFEST = Path("/content/m5-colab-pilot.approved.json")
WORK = Path("/content/atlasrank-m5-work")
INPUTS = WORK / "input"
VENV = WORK / "venv"
RECOVER_INTERRUPTED = False  # True only after confirming the prior Colab VM stopped.
if ALLOCATION_STARTED_UTC is None:
    raise PermissionError(
        "Record actual L4 allocation time and obtain one execution approval first"
    )

## Runtime discovery

This checks the native Colab Torch/CUDA build before installing anything. It rejects a non-L4, changed Torch build, insufficient RAM, or low free VRAM. Do not replace Colab's CUDA-enabled Torch.



In [ ]:
import os

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import json
import sys

import psutil
import torch

torch.cuda.init()
free_vram, total_vram = torch.cuda.mem_get_info(0)
native_torch_path = str(torch.__file__)
discovery = {
    "python": sys.version,
    "torch": str(torch.__version__),
    "torch_cuda": torch.version.cuda,
    "torch_path": native_torch_path,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0),
    "free_vram_bytes": free_vram,
    "total_vram_bytes": total_vram,
    "available_ram_bytes": psutil.virtual_memory().available,
}
print(json.dumps(discovery, indent=2))
if (
    sys.version_info[:2] != (3, 13)
    or str(torch.__version__) != "2.11.0+cu130"
    or torch.version.cuda != "13.0"
    or discovery["gpu"] != "NVIDIA L4"
    or not discovery["cuda_available"]
    or discovery["available_ram_bytes"] < 16 * 1024**3
    or free_vram < 8 * 1024**3
):
    raise RuntimeError("Runtime differs from approved M5 pilot; stop and report discovery")

In [ ]:
import hashlib
import json
import zipfile

approved = json.loads(APPROVED_MANIFEST.read_text())
if approved.get("execution_approved") is not True:
    raise PermissionError("The proposed manifest cannot execute training")
if (
    approved.get("input_archive_sha256")
    != "339434c0c6f7be24a29d50712c67cb98417c3b23223a10eb6f4042b066ff28ed"
):
    raise ValueError("Approved manifest does not pin the reviewed local training export")
if ARCHIVE_UPLOAD.stat().st_size != approved.get("input_archive_bytes"):
    raise ValueError("Training export size differs from the approved manifest")
digest = hashlib.sha256()
with ARCHIVE_UPLOAD.open("rb") as stream:
    for block in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(block)
if digest.hexdigest() != approved["input_archive_sha256"]:
    raise ValueError("Training ZIP checksum differs from the approved pin")
if approved.get("gpu_name") != "NVIDIA L4" or approved.get("batch_size") != 16:
    raise ValueError("Manifest device/batch configuration differs")
with zipfile.ZipFile(ARCHIVE_UPLOAD) as archive:
    bootstrap = archive.read("source/src/atlasrank/retrieval/colab_transport.py")
    if archive.testzip() is not None:
        raise ValueError("Training input ZIP failed CRC validation")
namespace = {"__name__": "atlasrank_m5_transport_bootstrap"}
exec(compile(bootstrap, "<reviewed-colab-transport>", "exec"), namespace)
WORK.mkdir(parents=True, exist_ok=True)
namespace["safe_extract"](ARCHIVE_UPLOAD, INPUTS, approved["upload_limit_bytes"])
export = json.loads((INPUTS / "EXPORT.json").read_text())
for relative, expected in export["source_sha256"].items():
    base = INPUTS if relative.startswith("environments/") else INPUTS / "source"
    candidate = base / relative
    actual = hashlib.sha256(candidate.read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f"Reviewed source/environment hash mismatch: {relative}")
sys.path.insert(0, str(INPUTS / "source/src"))
print("Verified training export:", approved["input_archive_sha256"])
print(
    "Train groups:",
    export["training_groups"],
    "selected products:",
    export["selected_product_identity_count"],
)

## Pinned overlay

The M4 Colab overlay is installed into a private work venv with system site packages and hashed wheels. It excludes Torch/NVIDIA packages. The notebook checks the native Torch version and file path remain unchanged.



In [ ]:
import subprocess

if not VENV.exists():
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(VENV)], check=True)
VENV_PYTHON = VENV / "bin/python"
requirements = INPUTS / "environments/colab/requirements.txt"
subprocess.run(
    [
        str(VENV_PYTHON),
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-cache-dir",
        "--no-deps",
        "--require-hashes",
        "--only-binary=:all:",
        "-r",
        str(requirements),
    ],
    check=True,
)
probe = subprocess.run(
    [
        str(VENV_PYTHON),
        "-c",
        "import torch,sys; print(sys.version); print(torch.__version__); "
        "print(torch.version.cuda); print(torch.__file__)",
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.splitlines()
print("Training environment:", probe)
if probe[1] != "2.11.0+cu130" or probe[2] != "13.0" or probe[3] != native_torch_path:
    raise RuntimeError("Pinned overlay changed or cannot see the native CUDA Torch build")

In [ ]:
import os
import shutil

from google.colab import drive

drive.mount("/content/drive")
PERSISTENT = Path(approved["persistent_destination"])
if not str(PERSISTENT).startswith("/content/drive/MyDrive/AtlasRank/m5/"):
    raise PermissionError("Only the approved private AtlasRank M5 Drive subtree is allowed")
if not approved.get("operator_owner_token"):
    raise PermissionError("Approved manifest lacks the exclusive runtime token")
inputs_drive = PERSISTENT / "inputs"
inputs_drive.mkdir(parents=True, exist_ok=True)
approved_copy = PERSISTENT / "approved-m5-pilot.json"
drive_archive = inputs_drive / "m5-training-input-v3.zip"
for source, target in ((APPROVED_MANIFEST, approved_copy), (ARCHIVE_UPLOAD, drive_archive)):
    expected = hashlib.sha256(source.read_bytes()).hexdigest()
    if target.exists():
        if hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise ValueError(f"Preserve existing Drive file; checksum differs: {target.name}")
    else:
        with target.open("xb") as destination, source.open("rb") as origin:
            shutil.copyfileobj(origin, destination)
os.environ["ATLAS_COLAB_EXCLUSIVE_RUNTIME"] = approved["operator_owner_token"]
print("Private destination:", PERSISTENT)
print("Input export persisted and hash-checked; package runner creates the owner lock.")

## The only training cell

Do not execute this before the single pilot approval. It chooses the latest immutable checkpoint only when resuming, validates its provenance and receipt, and starts at its consumed-batch cursor. The active-session lock prevents simultaneous Colab writers. On interruption, keep all artifacts, stop the prior runtime, then set RECOVER_INTERRUPTED=True in a new runtime.



In [ ]:
import json

checkpoints = PERSISTENT / "checkpoints"
checkpoint_files = sorted(
    (path for path in checkpoints.rglob("step-*.pt") if path.with_suffix(".json").is_file()),
    key=lambda path: int(path.stem.removeprefix("step-")),
)
resume_path = checkpoint_files[-1] if checkpoint_files else None
if RECOVER_INTERRUPTED and resume_path is None:
    raise RuntimeError("No checkpoint is available to resume")
command = [
    str(VENV_PYTHON),
    str(INPUTS / "source/scripts/run_m5_training.py"),
    "--archive",
    str(ARCHIVE_UPLOAD),
    "--manifest",
    str(APPROVED_MANIFEST),
    "--model",
    str(INPUTS / "model"),
    "--input-root",
    str(INPUTS),
    "--checkpoints",
    str(checkpoints),
    "--persistent",
    str(PERSISTENT),
    "--allocation-started-utc",
    str(ALLOCATION_STARTED_UTC),
]
if resume_path is not None:
    command.extend(["--resume", str(resume_path)])
if RECOVER_INTERRUPTED:
    command.append("--recover-interrupted")
run_env = os.environ.copy()
run_env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
run_env["PYTHONPATH"] = str(INPUTS / "source/src")
subprocess.run(command, check=True, env=run_env, cwd=WORK)

After the command, verify the session run JSON, end receipt, published-file receipt, and returned ZIP in the approved private Drive folder. The return bundle contains fine-tuned weights only if all 574 updates complete. Disconnect the L4 runtime promptly after evidence is saved. Never delete failed checkpoints or restart with a new owner/destination to reset the cap.

On interruption: preserve all files, stop the prior VM, use the same approved manifest and owner, record the new runtime allocation timestamp, re-upload the identical export if needed, and set RECOVER_INTERRUPTED=True. The package charges the prior allocation conservatively and refuses a concurrent writer. A pilot pass validates training mechanics; it does not establish retrieval quality until the full product catalog is re-embedded and Track A/B are evaluated on development only.
